# Shorts Maker

Runs entirely on Colab's disk/GPU -- nothing touches your laptop except the final clip, if you choose to download it.

**Before running:** set Runtime -> Change runtime type -> GPU (T4 is fine), so transcription runs fast.

Steps: 1) install deps, 2) pull code from GitHub, 3) paste a YouTube URL, 4) download + transcribe, 5) read the transcript and pick your clip's start/end time, 6) render with burned-in captions, 7) save to Drive or download.

## 1. Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q yt-dlp faster-whisper ffmpeg-python

## 2. Pull the pipeline code from your GitHub repo
Replace the URL below with your own repo once you've pushed it.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/shorts-maker.git"  # <-- change this

import os
if not os.path.exists("shorts-maker"):
    !git clone -q {REPO_URL}
else:
    !cd shorts-maker && git pull -q

import sys
sys.path.append("shorts-maker/src")

## 3. Paste your YouTube URL and download

In [ ]:
from downloader import download_video

VIDEO_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # <-- paste your link

video_path, info = download_video(VIDEO_URL, output_dir="downloads", max_height=720)
print("Downloaded:", video_path)
print("Title:", info.get("title"))
print("Duration (s):", info.get("duration"))

## 4. Transcribe with word-level timestamps

In [ ]:
from transcriber import transcribe, save_transcript, print_segments

# model_size: tiny/base/small/medium/large-v3 -- "small" is a good speed/accuracy balance on a free T4
transcript = transcribe(video_path, model_size="small", device="cuda", compute_type="float16")
save_transcript(transcript, "transcript.json")
print_segments(transcript)

## 5. Pick your clip(s)
Read the printed transcript above and list every moment you want turned into a short -- one dict per clip, `start`/`end` in seconds. Keep each clip under ~90s.

`style` is optional per clip (defaults to `bold_yellow` if omitted). Available styles: `bold_yellow`, `clean_white`, `pink_pop_top`, `hype_red` -- see `src/subtitles.py` `STYLE_PRESETS` to tweak or add your own.

In [ ]:
CLIPS = [
    {"start": 78.0,  "end": 125.0, "style": "bold_yellow"},
    {"start": 300.0, "end": 340.0, "style": "clean_white"},
    # add as many as you want
]

## 6. Generate captions and render all clips

In [ ]:
from clipper import render_clips

output_paths = render_clips(video_path, CLIPS, transcript["words"], output_dir="output", vertical=True)

## 7. Preview a clip
Change the index (0 = first clip, 1 = second, etc.) to preview a different one.

In [ ]:
from IPython.display import Video
Video(output_paths[0], embed=True, width=360)

## 8. Download all clips

In [ ]:
from google.colab import files
for p in output_paths:
    files.download(p)

## 9. Cleanup (free Colab's disk)
Run this after you're done cutting all the clips you want from this video.

In [ ]:
from downloader import cleanup_video
cleanup_video(video_path)
print("Source video deleted from Colab disk.")